# Commodity Open 2026: the shipped baselines, validated and scored

This notebook runs the five baselines of RULES.md sections 3 and 4 over the holdout, writes their signal files in the submission format, checks them with the validator and scores them with the shipped scorer.

They are the baselines the report puts beside an entry, and they are not reimplemented here: `swing_dataset.kit.baselines` is a wrapper over `swing_dataset.scoring2.baselines`, so what you read is what you are ranked against.

| challenge | name | rule |
|---|---|---|
| 1 | `range_break` | the prior session settled above the high or below the low of the session before it |
| 1 | `stretch_reversal` | the chain is past the fitted 80th percentile of its own 20-session stretch, so lean the other way |
| 2 | `prevailing_direction` | section 4's null: the sign of the last five completed sessions' move |
| 2 | `rv_band` | the sign of the 20-session trend, with a realized-volatility band |
| 2 | `range_break_vol` | the range break with the volatility-scaled target and band |

All of them are loops: every instant refits its own quantities on rows published before it, and nothing is fitted once and reused. They clear no prize.

You need the pack (a directory with `MANIFEST.json`) and the `swing-dataset` package. Point `SWING_PACK_ROOT` at the pack, or place it at `../pack`.

In [1]:
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd

from swing_dataset.kit import THRESHOLD_PCT, formats, loop, repro
from swing_dataset.kit import baselines as B
from swing_dataset.scoring2 import RULES_VERSION
from swing_dataset.scoring2 import score as S

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)

PACK_ROOT = Path(os.environ.get("SWING_PACK_ROOT", "../pack")).resolve()
OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)
assert (PACK_ROOT / "MANIFEST.json").exists(), "set SWING_PACK_ROOT to the directory that holds MANIFEST.json"

# Task 1's catch rate is counted against the realized-swing inventory, swings/swings.parquet in the pack.
SWINGS = Path(os.environ.get("SWING_SWINGS", "")) if os.environ.get("SWING_SWINGS") else repro.swings_path(PACK_ROOT)
HOLDOUT = repro.period_from("holdout")
print("pack:", PACK_ROOT)
print("holdout:", HOLDOUT.to_dict())
print("swing inventory:", SWINGS, "present" if SWINGS.exists() else "MISSING -- Task 1 cannot be scored without it")
print("thresholds (percent of the price at the turn):", THRESHOLD_PCT)

pack: pack/v2.1
holdout: {'name': 'holdout', 'start': '2026-06-01', 'end': '2026-12-31', 'train_end_utc': '2026-06-01T03:00:00+00:00'}
swing inventory: pack/v2.1/swings/swings.parquet present
thresholds (percent of the price at the turn): {'gold': 1.5, 'silver': 2.0, 'copper': 1.5, 'crude': 3.0}


## The price path and the feature chain

Everything a baseline needs is two objects the scorer already builds: `PriceSeries`, the designated contract's bars with a lookup at any instant, and `FeatureChains`, the roll-free chain the 20-session trend and realized volatility come from. Both take frames, so `kit.loop.price_view` hands them what the point-in-time client returned.

Reading the whole history here is not a leak. Every lookup a baseline makes is bounded by the instant it is deciding at: `known_pos` finds the last session *published* before it and `matured_slice` the outcomes that had *matured* by then. `template/run.py` does the same thing through the client, with a read log the organizers check; this notebook takes the short route and the last cell shows the two agree.

In [2]:
from swing_dataset.scoring.chain import FeatureChains
from swing_dataset.scoring2.prices import PriceSeries

t0 = time.perf_counter()
prices = formats.load_sessions(PACK_ROOT)
chains = FeatureChains.from_parquet(PACK_ROOT / "mcx" / "mcx_daily.parquet")
window = S.resolve_window("holdout", prices)
print(f"built in {time.perf_counter() - t0:.1f}s")
print("window:", window.to_dict())
pd.DataFrame(
    [
        {
            "asset": a,
            "sessions in the window": len(prices.sessions_between(a, window.start, window.end)),
            "first": prices.sessions_between(a, window.start, window.end)[0],
            "last": prices.sessions_between(a, window.start, window.end)[-1],
            "threshold %": THRESHOLD_PCT[a],
        }
        for a in prices.assets()
    ]
).set_index("asset")

built in 5.7s
window: {'period': 'holdout', 'start_utc': '2026-06-01T00:00:00+00:00', 'end_utc': '2026-10-23T18:00:00+00:00'}


,sessions in the window,first,last,threshold %
asset,,,,
copper,104,2026-06-01,2026-10-23,1.5
crude,104,2026-06-01,2026-10-23,3.0
gold,104,2026-06-01,2026-10-23,1.5
silver,104,2026-06-01,2026-10-23,2.0


## Run the five baselines

`B.run_baseline` is `scoring2.baselines.run`: for every asset and every session of the window it asks the rule at that session's pre-open, and writes a signal row per horizon it signals on, or one `no_signal` row with a reason when it signals on neither. That is the shape the coverage floor of section 5 is written for: a decision record on at least 90 percent of each asset's sessions, where a decision record is a signal **or** an explicit `no_signal`.

In [3]:
signal_files = {}
rows = []
for challenge in (1, 2):
    for rule in B.baselines_for(challenge):
        t0 = time.perf_counter()
        frame = B.run_baseline(rule, prices, chains, window)
        path = formats.write_predictions(frame, OUT_DIR / f"baseline_{challenge}_{rule.name}_holdout.parquet")
        signal_files[(challenge, rule.name)] = path
        summary = formats.summarize(frame)
        rows.append(
            {
                "challenge": challenge,
                "baseline": rule.name,
                "signals": summary["signals"],
                "no_signal": summary["no_signals"],
                "up": summary["by_direction"].get("up", 0),
                "down": summary["by_direction"].get("down", 0),
                "seconds": round(time.perf_counter() - t0, 1),
            }
        )
pd.DataFrame(rows).set_index(["challenge", "baseline"])

signals  no_signal   up  down  seconds
challenge baseline                                                    
1         range_break               468        182  234   234      0.3
          stretch_reversal          150        341   74    76      0.3
2         prevailing_direction      832          0  390   442      0.6
          rv_band                   832          0  362   470      0.5
          range_break_vol           468        182  234   234      0.5

In [4]:
frame = formats.read_predictions(signal_files[(1, "range_break")])
print("a signal row and a no_signal row of range_break:")
display(frame[frame["signal_utc"].notna()].head(2))
display(frame[frame["decision_utc"].notna()].head(2))

a signal row and a no_signal row of range_break:


,asset,signal_utc,direction,horizon,target_pct,invalidation_pct,confidence,evidence,decision_utc,reason,baseline,fallback
1,copper,2026-06-02 03:00:00+00:00,up,5,1.845898,1.5,0.5,range break: settlement 1365.8 against the pri...,NaT,NaN,range_break,False
2,copper,2026-06-02 03:00:00+00:00,up,10,3.012560,1.5,0.5,range break: settlement 1365.8 against the pri...,NaT,NaN,range_break,False


,asset,signal_utc,direction,horizon,target_pct,invalidation_pct,confidence,evidence,decision_utc,reason,baseline,fallback
0,copper,NaT,NaN,<NA>,NaN,NaN,NaN,NaN,2026-06-01 03:00:00+00:00,settlement 1348.75 inside the prior range 1343...,range_break,False
5,copper,NaT,NaN,<NA>,NaN,NaN,NaN,NaN,2026-06-04 03:00:00+00:00,settlement 1367.9 inside the prior range 1364 ...,range_break,False


## Validate

`tools/validate_submission.py` and `formats.validate` hand every row to `scoring2.signals.normalize`, so a file the validator calls valid is one the scorer calls valid. With the pack it also measures the coverage floor.

The two things that catch people out: `evidence` is required, and an invalid row stays in the denominator and can never be correct, so a row the rules reject costs a submission rather than disappearing from it.

In [5]:
reports = {}
for (challenge, name), path in signal_files.items():
    report = formats.validate_file(path, challenge, pack_root=PACK_ROOT)
    reports[(challenge, name)] = report
print(reports[(1, "range_break")].render())
print()
print(reports[(2, "rv_band")].render())

signal file: output/baseline_1_range_break_holdout.parquet
challenge: 1
rows: 650  signals: 468 (468 valid, 0 invalid)  no_signal: 182 (182 valid)
horizons: h5=234 h10=234
assets: copper=112 crude=116 gold=120 silver=120
coverage (floor 0.90, per asset): copper=1.000 crude=1.000 gold=1.000 silver=1.000 -> met
errors: 0  warnings: 0
result: VALID

signal file: output/baseline_2_rv_band_holdout.parquet
challenge: 2
rows: 832  signals: 832 (830 valid, 2 invalid)  no_signal: 0 (0 valid)
horizons: h5=416 h10=416
assets: copper=208 crude=208 gold=208 silver=208
coverage (floor 0.90, per asset): copper=1.000 crude=1.000 gold=1.000 silver=1.000 -> met
error: band_too_wide: p90_pct - p10_pct is more than BAND_WIDTH_CAP = 12 asset thresholds wide; a band that contains every move scores depth without saying anything (x2)
errors: 1  warnings: 0
result: INVALID


Two `rv_band` rows are rejected as `band_too_wide`, and that is the cap doing its job rather than a bug: both are h = 10 bands whose upper end came from a trailing excursion quantile taken in June. Over the three shipped Task 2 baselines the cap costs 0.29, 0.29 and 0.51 percent of their rows, six of 1,750 (`FORMATS2.md` gives the figures for the window to 14 September as well). They stay in the denominator and can never be realized, which is what an invalid row costs.

In [6]:
pd.DataFrame(
    [
        {
            "challenge": c,
            "baseline": n,
            "valid signals": r.valid_signal_rows,
            "invalid": r.invalid_signal_rows,
            "no_signal": r.valid_no_signal_rows,
            "coverage met": r.coverage["met"],
            "pooled coverage": round(r.coverage["pooled"]["share"], 4),
            "result": "VALID" if r.ok else "INVALID",
        }
        for (c, n), r in reports.items()
    ]
).set_index(["challenge", "baseline"])

valid signals  invalid  no_signal  coverage met  pooled coverage   result
challenge baseline                                                                                       
1         range_break                     468        0        182          True              1.0    VALID
          stretch_reversal                150        0        341          True              1.0    VALID
2         prevailing_direction            830        2          0          True              1.0  INVALID
          rv_band                         830        2          0          True              1.0  INVALID
          range_break_vol                 466        2        182          True              1.0  INVALID

## What an invalid row looks like

A band wide enough to contain every move would score depth full marks without saying anything, so section 4 caps it at `BAND_WIDTH_CAP` asset thresholds. Here is what happens to a file that tries it.

In [7]:
from swing_dataset.scoring2 import BAND_WIDTH_CAP

cheat = formats.read_predictions(signal_files[(2, "rv_band")]).copy()
cheat.loc[cheat["signal_utc"].notna(), ["p10_pct", "p90_pct"]] = [-99.0, 99.0]
report = formats.validate(cheat, 2)
print(f"BAND_WIDTH_CAP = {BAND_WIDTH_CAP} thresholds, so at most {BAND_WIDTH_CAP * THRESHOLD_PCT['gold']:g} points wide on gold")
print(report.render())

BAND_WIDTH_CAP = 12 thresholds, so at most 18 points wide on gold
signal file: <frame>
challenge: 2
rows: 832  signals: 832 (0 valid, 832 invalid)  no_signal: 0 (0 valid)
horizons: h5=416 h10=416
assets: copper=208 crude=208 gold=208 silver=208
error: band_too_wide: p90_pct - p10_pct is more than BAND_WIDTH_CAP = 12 asset thresholds wide; a band that contains every move scores depth without saying anything (x832)
error: nothing_valid: no row in the file is a decision record the rules recognise
errors: 2  warnings: 0
result: INVALID


## Score with the shipped scorer

`repro.evaluate` runs `python -m swing_dataset.scoring2 evaluate --baselines` over the holdout. Task 1 returns catch rate, correctness and depth; Task 2 returns direction, depth and improvement. Each is a percentage, the success rate is their plain average, and every one carries a 90 percent episode-block interval from 2000 seeded resamples.

Scoring a baseline against itself gives an improvement of zero by construction, which is what the report says about it: the improvement score exists to measure a learning model against its own frozen copy, and a baseline has nothing to freeze.

In [8]:
t0 = time.perf_counter()
c1_report = None
if SWINGS.exists():
    c1_report = repro.evaluate(
        PACK_ROOT,
        signal_files[(1, "range_break")],
        challenge=1,
        period=HOLDOUT,
        out=OUT_DIR / "report_1_holdout.json",
        swings=SWINGS,
        baselines=True,
    )
    print(repro.render_report(c1_report))
else:
    print(f"no realized-swing inventory at {SWINGS}: the catch rate cannot be counted, so Task 1 is not scored here.")
    print("set SWING_SWINGS to the inventory (swings/swings.parquet in the pack) and re-run this cell.")
print(f"\n{time.perf_counter() - t0:.1f}s")

Task 1, holdout window 2026-06-01 to 2026-10-23, horizons [5, 10]
  catch rate 24.8 [21.0, 28.9]  correctness 51.9 [47.1, 56.6]  depth 65.1 [58.6, 72.0]
  success rate 47.3 [43.7, 50.9] (minimum 70)
  signals 468, invalid 0, unmatured 2, scored 466, undecided 19 (4.1 percent of scored, left out of correctness), no_signal 182, swings 230, caught 57
  coverage (floor 0.90): copper=1.000 crude=1.000 gold=1.000 silver=1.000 -> met
  ten large swings: 10 of 10 caught, at least 7 needed on the holdout
     1. crude  down 2026-06-10T03:30 to 2026-06-16T03:30  -18.93 percent, 6.31 thresholds  caught by the signal of 2026-06-15T03:00:00
     2. gold   up   2026-08-03T03:30 to 2026-08-12T13:30  +9.29 percent, 6.19 thresholds  caught by the signal of 2026-08-05T03:00:00
     3. crude  up   2026-09-04T13:30 to 2026-09-11T03:30  +18.13 percent, 6.04 thresholds  caught by the signal of 2026-09-08T03:00:00
     4. crude  down 2026-07-23T17:30 to 2026-07-28T16:30  -17.59 percent, 5.86 thresholds  caug

In [9]:
t0 = time.perf_counter()
c2_report = repro.evaluate(
    PACK_ROOT,
    signal_files[(2, "rv_band")],
    challenge=2,
    period=HOLDOUT,
    out=OUT_DIR / "report_2_holdout.json",
    frozen=signal_files[(2, "rv_band")],
    swings=SWINGS if SWINGS.exists() else None,
    baselines=True,
)
print(repro.render_report(c2_report))
print(f"\n{time.perf_counter() - t0:.1f}s")

Task 2, holdout window 2026-06-01 to 2026-10-23, horizons [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
  direction 51.2 [45.9, 56.8]  depth 67.3 [58.2, 76.5]  improvement 0.0 [0.0, 0.0]
  success rate 39.5 [36.6, 42.4] (minimum 70)
  signals 832, invalid 2, unmatured 2, scored 830, no_signal 0
  coverage (floor 0.90): copper=1.000 crude=1.000 gold=1.000 silver=1.000 -> met
  hard conditions: learning_beats_frozen no, second_half_at_least_first_half yes, direction_at_least_minimum no, frozen_beats_prevailing_null yes, learning_beats_prevailing_null yes
  prevailing-direction null at the same instants: 46.9
  baseline prevailing_direction: success rate 42.3 [39.5, 45.0]
  baseline rv_band: success rate 39.5 [36.6, 42.4]
  baseline range_break_vol: success rate 39.8 [36.3, 42.9]
  minimum on the holdout period: NOT met (both periods and the section 4a gate decide the prize)

14.0s


In [10]:
def score_table(report):
    names = ("catch_rate", "correctness", "depth") if report["challenge"] == 1 else ("direction", "depth", "improvement")
    rows = []
    for who, block in [("this file", report)] + sorted((report.get("baselines") or {}).items()):
        scores = block["scores"]
        row = {"who": who}
        for name in (*names, "success_rate"):
            entry = scores.get(name) or {}
            row[name.replace("_", " ")] = None if entry.get("value") is None else round(entry["value"], 1)
        rows.append(row)
    return pd.DataFrame(rows).set_index("who")

if c1_report is not None:
    print("Task 1, holdout")
    display(score_table(c1_report))
print("Task 2, holdout")
score_table(c2_report)

Task 1, holdout


,catch rate,correctness,depth,success rate
who,,,,
this file,24.8,51.9,65.1,47.3
range_break,24.8,51.9,65.1,47.3
stretch_reversal,7.4,43.8,46.0,32.4


Task 2, holdout


,direction,depth,improvement,success rate
who,,,,
this file,51.2,67.3,0.0,39.5
prevailing_direction,46.9,79.9,0.0,42.3
range_break_vol,49.6,69.7,0.0,39.8
rv_band,51.2,67.3,0.0,39.5


## The null, and the conditions that cannot be averaged away

Section 4 prints the always-prevailing-direction null beside every Task 2 entry: the sign of the last five completed sessions' move, issued at the entry's own instants. An entry whose direction score does not exceed it is not eligible, however good the other two scores are. Four more conditions are hard in the same way, and the report states each as a boolean.

In [11]:
print("prevailing-direction null:", json.dumps(c2_report["prevailing_direction_null"], indent=2))
print()
print("paired direction (the rows the file and its frozen copy both issued):", c2_report["paired_direction"])
print("halves:", c2_report["halves"])
print()
for name, value in c2_report["award_conditions"]["hard_conditions"].items():
    print(f"  {name}: {value}")

prevailing-direction null: {
  "rule": "the sign of the last five completed sessions' move, issued at the entry's instants with the entry's horizon, expected and invalidation at the asset threshold",
  "direction_at_entry_instants": 46.85990338164252,
  "direction_at_frozen_instants": 46.85990338164252,
  "n": 828
}

paired direction (the rows the file and its frozen copy both issued): {'n': 830, 'learning': 51.204819277108435, 'frozen': 51.204819277108435}
halves: {'first_half_direction': 48.55769230769231, 'second_half_direction': 53.864734299516904, 'first_half_sessions': 52, 'second_half_sessions': 52}

  learning_beats_frozen: False
  second_half_at_least_first_half: True
  direction_at_least_minimum: False
  frozen_beats_prevailing_null: True
  learning_beats_prevailing_null: True


## The same rows, through the point-in-time client

The baselines above read a price series built from the whole pack, bounded at each instant by the rule's own lookups. A submission may not do that: `run.py` reads through `PointInTime`, which returns only rows available strictly before the instant asked for and logs every read.

The two routes produce the same rows. Ten sessions is enough to show it.

In [12]:
from datetime import date

from swing_dataset.pack.pit import PointInTime, ReadLog

start, end = date(2026, 7, 1), date(2026, 7, 14)
log_path = OUT_DIR / "pit_crosscheck.readlog.jsonl"
pit = PointInTime(PACK_ROOT, read_log=ReadLog(log_path))
t0 = time.perf_counter()
result = loop.run(B.BaselineModel("range_break"), pit, challenge=1, period=repro.period_from("holdout", start, end))
print(f"{len(result.sessions)} sessions through the client in {time.perf_counter() - t0:.1f}s, {result.decisions.shape[0]} rows")

key = ["asset", "signal_utc", "direction", "horizon"]
fast = formats.read_predictions(signal_files[(1, "range_break")])
fast = fast[fast["signal_utc"].between(pd.Timestamp(loop.preopen_instant(start)), pd.Timestamp(loop.preopen_instant(end)))]
fast = fast.sort_values(key).reset_index(drop=True)
through_client = result.signals.sort_values(key).reset_index(drop=True)
assert len(fast) == len(through_client) > 0
assert fast["direction"].tolist() == through_client["direction"].tolist()
assert np.allclose(fast["target_pct"].to_numpy(float), through_client["target_pct"].to_numpy(float), atol=1e-12)
print("the two routes agree on every row")

check = repro.check_read_log(ReadLog.read(log_path), result.decisions, HOLDOUT.train_end)
print("read log:", check["entries"], "reads,", len(check["violations"]), "violations; files:", sorted(check["files"]))

10 sessions through the client in 15.5s, 63 rows
the two routes agree on every row
read log: 69 reads, 0 violations; files: ['labels/designations', 'mcx/mcx_daily', 'mcx/mcx_sessions']


In [13]:
summary = {
    "rules_version": RULES_VERSION,
    "pack": str(PACK_ROOT),
    "window": c2_report["window"],
    "swings": str(SWINGS) if SWINGS.exists() else None,
    "challenge_1": None
    if c1_report is None
    else {
        "scored_file": str(signal_files[(1, "range_break")]),
        "scores": c1_report["scores"],
        "counts": c1_report["counts"],
        "coverage": c1_report["coverage"],
        "baselines": {n: b["scores"] for n, b in c1_report["baselines"].items()},
    },
    "challenge_2": {
        "scored_file": str(signal_files[(2, "rv_band")]),
        "scores": c2_report["scores"],
        "counts": c2_report["counts"],
        "coverage": c2_report["coverage"],
        "hard_conditions": c2_report["award_conditions"]["hard_conditions"],
        "prevailing_direction_null": c2_report["prevailing_direction_null"],
        "baselines": {n: b["scores"] for n, b in c2_report["baselines"].items()},
    },
}
(OUT_DIR / "baseline_scores.json").write_text(json.dumps(summary, indent=2, default=str) + "\n")
print("wrote", OUT_DIR / "baseline_scores.json")
sorted(p.name for p in OUT_DIR.iterdir())

wrote output/baseline_scores.json


['baseline_1_range_break_holdout.parquet',
 'baseline_1_stretch_reversal_holdout.parquet',
 'baseline_2_prevailing_direction_holdout.parquet',
 'baseline_2_range_break_vol_holdout.parquet',
 'baseline_2_rv_band_holdout.parquet',
 'baseline_scores.json',
 'pit_crosscheck.readlog.jsonl',
 'report_1_holdout.json',
 'report_2_holdout.json']

## From here to a submission

`template/` is a complete submission: `run.py` drives this loop through the point-in-time client and writes the read log, and `model.py` holds a worked model for each challenge. Copy the directory, replace `model.py`, keep the three methods, and follow `template/README.md`.

The template's own scores are far under the 70 percent minimum. That is the point of a worked example: it shows the mechanics, the file format, the coverage floor and the learning loop, and leaves the edge to you.